## Data Cleaning/Processing

### Load libraries and data

In [24]:
import pandas as pd

df = pd.read_csv("../data/raw/housing.csv")

print(df.shape)
print(df.dtypes.value_counts())
print("\nMissing values (columns with nulls only):")
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
#split missing into numeric vs categorical
numeric_missing = df.select_dtypes(include=['int64', 'float64']).isnull().sum()
numeric_missing = numeric_missing[numeric_missing > 0].sort_values(ascending=False)
categorical_missing = df.select_dtypes(include=['object']).isnull().sum()
categorical_missing = categorical_missing[categorical_missing > 0].sort_values(ascending=False)

print(missing)
print("Numeric columns with missing values:")
print(numeric_missing)

print("\nCategorical columns with missing values:")
print(categorical_missing)

(1460, 81)
object     43
int64      35
float64     3
Name: count, dtype: int64

Missing values (columns with nulls only):
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64
Numeric columns with missing values:
LotFrontage    259
GarageYrBlt     81
MasVnrArea       8
dtype: int64

Categorical columns with missing values:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
GarageFinish      81
GarageQual        81
GarageType        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtFinType1      37
BsmtCond          37
El

### Data Observations
Group 1:

    PoolQC       1453 / 1460 missing = 99.5%
    MiscFeature  1406 / 1460 missing = 96.3%
    Alley        1369 / 1460 missing = 93.8%
    Fence        1179 / 1460 missing = 80.8%

These columns are so sparse that they can't teach the model anything meaningful. For `PoolQC`, `Alley`, and `Fence` it could mean that they dont have it but the data is so small that keeping them wouldn't make a substantial difference in training the model

Decision: Drop these

Group 2:

    MasVnrType    872 missing
    FireplaceQu   690 missing
    GarageType     81 missing
    GarageFinish   81 missing
    GarageQual     81 missing
    GarageCond     81 missing
    BsmtExposure   38 missing
    BsmtFinType2   38 missing
    BsmtQual       37 missing
    BsmtCond       37 missing
    BsmtFinType1   37 missing

For these columns, a missing value almost certainly means that the house does not have the feature so `"None"` would be a good replacement for `Null` and then it can be encoded

Group 3:

    LotFrontage  259 missing
    GarageYrBlt   81 missing
    MasVnrArea     8 missing

`LotFrontage` - linearfeet of street connected to the property. Missing likely means that it wasn't recorded and not zero. **Median imputation** would be the correct choice since this datas distribution is right skewed


`GarageYrBlt` - Of the house has no garage (Which was established that 81 houses had no garage (group2) ) this should just be 0 and not median

`MasVnrArea` - Only 8 missing, pirs with `MasVnrType`. Fill with 0 since no masonary venner means area is zero.

Group 4:

    Electrical  1 missing

One row is missing just filll it with the mode (most frequent value).

In [ ]:
#Old Missing:
print(f"Old\n{missing}")
# 1. Drop High- Sparsity columns
for col, count in missing.items():
    if count > 1000:
        df = df.drop(columns=col)
missing = df.isnull().sum()
numeric_missing = df.select_dtypes(include=['int64', 'float64']).isnull().sum()
categorical_missing = df.select_dtypes(include=['object']).isnull().sum()

# 2. Fill categorical "None" columns
for col, count in categorical_missing.items():
    if count in range (30, 900):
        df[col] = df[col].fillna('None')
missing = df.isnull().sum()
categorical_missing = df.select_dtypes(include=['object']).isnull().sum()

# Fill numeric columns
df = df.fillna({
    'LotFrontage': df['LotFrontage'].median(),
    'GarageYrBkt': 0,
    'MasVnrArea' : 0
    })

# 4. Fill Electrical with Node
df = df.fillna({'Electrical': df['Electrical'].mode()[0]})

missing = df.isnull().sum()
print(f"New\n{missing}")

Old
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64
New
Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
SalePrice        0
Length: 77, dtype: int64


In [22]:
import sys
sys.path.insert(0, '../src')
from preprocessing import impute_missing

df_clean = impute_missing(df)
df_clean.to_csv('../data/processed/housing_clean.csv', index=False)